In [ ]:
from tokenizers import Tokenizer      #holds the tokenizers
from tokenizers.models import BPE           # the actual BPE models
from tokenizers.trainers import BpeTrainer         # learns the vocabulary + merge rules
from tokenizers.pre_tokenizers import ByteLevel       # handles word boundries before bpe
from pathlib import Path                                #handle file paths
from tokenizers.decoders import ByteLevel as ByteLevelDecoder       #decodes byte-level tokens

In [ ]:
#dataset paths
train_dir = Path("/srv/data/lt2326-h26/a1/train")
languages = ["en", "tr", "zh"]

In [ ]:
#training files
train_files = []

for lang in languages:
   path = train_dir / f"{lang}.txt"
   train_files.append(str(path))             #add each language file to the list of training files

In [ ]:
#creating BPE-10k tokenizer

tokenizer = Tokenizer(BPE(unk_token="<UNK>"))    #tokenizer object with BPE model and unknown token but it is not trained yet 
tokenizer.pre_tokenizer = ByteLevel()               #use byte-level tokenization before BPE

In [ ]:
#creating BPE trainer
trainer = BpeTrainer(
    vocab_size=10000,      #building a vocab of about 10k tokens
    special_tokens=["<UNK>"],           #add unknown token to the vocab
    initial_alphabet=ByteLevel.alphabet()           #include complete byte-level alphabet in the vocab
)


In [ ]:
tokenizer.train(train_files, trainer)     #train tokenizer on combined EN + TR + ZH data


In [ ]:
#checking vocabulary size and first 15 tokens

vocab = tokenizer.get_vocab()

print("Vocabulary size:", tokenizer.get_vocab_size())
print("First 30 vocabulary tokens:")
vocab_by_id = sorted(vocab.items(), key=lambda x: x[1])     # sort vocabulary by token ID
for token, token_id in vocab_by_id[:15]: # print first 15 tokens
    print(f"ID: {token_id},Token: {token}")

In [ ]:
#Encode /Decode test

tokenizer.decoder = ByteLevelDecoder()   #add a decoder to the tokenizer so that we can decode the tokenized text back to original text

#encode / decode test
test_texts = [
    "She played in two matches at this championship.",
    "Germantown akademisine devam ettikten sonra 1869-1870 yılları Fransa ve Almanya'da okullarda geçti.",
    "，也是一所专门提供研究生及以上级别课程的私立精英级大学。"
]

for text in test_texts:
    encoding = tokenizer.encode(text)           # convert text into token IDS
    decoded = tokenizer.decode(encoding.ids)        #convert token IDS back to text

    print("\n========== Encode / Decode Test ==========")
    print("Original:", text)
    print("Tokens:", encoding.tokens)
    print("Decoded:", decoded)
    print("Match:", text == decoded.lstrip())  # Remove trailing whitespace for comparison

In [ ]:
#validation statistics

valid_dir = Path("/srv/data/lt2326-h26/a1/valid")


#example tokenization on validation data
for lang in languages:
    path = valid_dir / f"{lang}.txt"

    text = path.read_text(encoding="utf-8")
    lines= text.splitlines()

    sentence = lines[0] #take the first sentence from the validation file for each language 
    encoding = tokenizer.encode(sentence)
    tokens = encoding.tokens

    print(f"\n========== {lang.upper()} ==========")
    print("Original text:", sentence)
    print("BPE Tokens:", tokens)
    print("Number of BPE Tokens:", len(tokens))


In [ ]:
#validation statistics for each language
for lang in languages:
    path = valid_dir / f"{lang}.txt"
    text = path.read_text(encoding="utf-8")
    lines= text.splitlines()

    total_bpe_tokens = 0
    for sentence in lines:
        encoding = tokenizer.encode(sentence)
        bpe_tokens = encoding.tokens
        total_bpe_tokens += len(bpe_tokens)         #count BPE tokens for each sentence and add to total

    total_characters = sum(len(sentence) for sentence in lines)         #count Unicode characters in each sentence and add to total

    print(f"\n========== {lang.upper()} ==========")
    print("Total number of BPE tokens:", total_bpe_tokens)
    print("Number of sentences:", len(lines))
    print("Total unicode characters:", total_characters)
    average_bpe_tokens = total_bpe_tokens / len(lines)
    print("Average number of BPE tokens per sentence:", average_bpe_tokens)
    print("BPE tokens per Unicode character:", total_bpe_tokens / total_characters)
    print("Average Unicode characters per token:", total_characters / total_bpe_tokens)


In [ ]:
#Encode / Decode and UNK sanity check

print("=" * 10 + "Encode/ Decode and UNK sanity check" + "=" * 10)

test_texts = "Hello World!"

encoding = tokenizer.encode(test_texts)
print("Original text:", test_texts)
print("BPE Tokens:", encoding.tokens)
print("Token IDs:", encoding.ids)

decoded = tokenizer.decode(encoding.ids)
print("Decoded text:", decoded)
print("Match:", test_texts == decoded.lstrip())  # Remove trailing whitespace for comparison

#checking whether the tokenizer produced <UNK>
unk_id = vocab.get("<UNK>")

contains_unk = unk_id in encoding.ids
print("Contains <UNK> token:", contains_unk)


In [ ]:
#Unseen character test

print("\n========== Unseen Character Test ==========")

test_text = "Hello 🙂"

encoding = tokenizer.encode(test_text)
print("Original text:", test_text)
print("BPE Tokens:", encoding.tokens)
print("Token IDs:", encoding.ids)

decoded = tokenizer.decode(encoding.ids)
print("Decoded text:", decoded)
print("Match:", test_text == decoded.lstrip())  # Remove trailing whitespace for comparison

#checking whether the tokenizer produced <UNK>
unk_id = vocab.get("<UNK>")

contains_unk = unk_id in encoding.ids
print("Contains <UNK> token:", contains_unk)


In [ ]:
#save trained tokenizer
output_dir = Path("tokenizers")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "bpe_10k.json"

tokenizer.save(str(output_path))            #saving tokenizers as JSON file so that it can be loaded later for inference or further training

print("=" * 10 + "Saving Tokenizer" + "=" * 10)
print("Tokenizer saved to:", output_path)


In [ ]:
#reloading the saved tokenizer
loaded_tokenizer = Tokenizer.from_file(str(output_path))

print("=" * 10 + "Reloading Test" + "=" * 10)

test_text = "She played in two matches at this championship."

encoding = loaded_tokenizer.encode(test_text)
decoded = loaded_tokenizer.decode(encoding.ids)

print("Original text:", test_text)
print("BPE Tokens:", encoding.tokens)
print("Decoded text:", decoded)
print("Match:", test_text == decoded.lstrip())  # checking whether the decoded text matches the original text after reloading the tokenizer
